In [ ]:
import os
import glob
import duckdb

def run_conversion():
    # Kaggle-specific paths
    input_base = r"D:\EmerG-NeT\openalex-works"
    output_base = r"D:\EmerG-NeT\Phase_1"
    
    # Path for the error log
    os.makedirs(output_base, exist_ok=True)
    error_log_path = os.path.join(output_base, "failed_files.txt")

    # Initialize DuckDB connection
    con = duckdb.connect()

    # Get all subfolders
    subfolders = sorted([f.path for f in os.scandir(input_base) if f.is_dir()])
    if not subfolders:
        subfolders = [input_base]

    for folder in subfolders:
        folder_name = os.path.basename(folder)
        
        # Mirror folder structure
        relative_folder = os.path.relpath(folder, input_base)
        output_folder = os.path.join(output_base, relative_folder)
        os.makedirs(output_folder, exist_ok=True)

        # Find files
        gz_files = glob.glob(os.path.join(folder, "*.gz"))
        if not gz_files:
            gz_files = glob.glob(os.path.join(folder, "part_*"))

        if not gz_files:
            continue

        print(f"\n📂 Processing folder: {folder_name}")

        for gz_file in gz_files:
            original_base = os.path.basename(gz_file)
            file_name = original_base.split('.')[0] + ".parquet"
            output_path = os.path.join(output_folder, file_name)

            print(f"   📄 Converting: {original_base}", end="\r")

            # 1. Removed the extra snapshot column from the SELECT statement
            query = f"""
            COPY (
                SELECT * FROM read_json_auto(
                    '{gz_file}', 
                    format='newline_delimited', 
                    sample_size=100000,
                    ignore_errors=True
                )
            ) TO '{output_path}' (FORMAT 'PARQUET', COMPRESSION 'SNAPPY');
            """

            try:
                con.execute(query)
                
                # 2. Delete the source file after successful conversion
                try:
                    os.remove(gz_file)
                except PermissionError:
                    # This will happen on Kaggle Input datasets
                    pass 
                except Exception as e:
                    print(f"\n⚠️ Could not delete {original_base}: {e}")

            except Exception as e:
                print(f"\n❌ Failed to convert: {gz_file}. Logging...")
                with open(error_log_path, "a") as f:
                    f.write(f"{gz_file} | Error: {str(e)}\n")

        print(f"\n✅ Finished folder: {folder_name}")

if __name__ == "__main__":
    run_conversion()


📂 Processing folder: updated_date=2025-07-23
   📄 Converting: part_0001.gz
✅ Finished folder: updated_date=2025-07-23

📂 Processing folder: updated_date=2025-07-24


In [ ]:
import os
import glob
import duckdb
from tqdm import tqdm

def run_conversion():
    # Kaggle-specific paths
    input_base = r"D:\EmerG-NeT\openalex-works"
    output_base = r"D:\EmerG-NeT\Phase_1"
    
    os.makedirs(output_base, exist_ok=True)
    error_log_path = os.path.join(output_base, "failed_files.txt")

    # Initialize DuckDB connection
    con = duckdb.connect()

    # Get all subfolders
    subfolders = sorted([f.path for f in os.scandir(input_base) if f.is_dir()])
    if not subfolders:
        subfolders = [input_base]

    # Outer Progress Bar: Subfolders
    pbar_folders = tqdm(subfolders, desc="📁 Total Progress", unit="folder")

    for folder in pbar_folders:
        folder_name = os.path.basename(folder)
        pbar_folders.set_description(f"📁 Processing: {folder_name}")
        
        # Mirror folder structure
        relative_folder = os.path.relpath(folder, input_base)
        output_folder = os.path.join(output_base, relative_folder)
        os.makedirs(output_folder, exist_ok=True)

        # Find files
        gz_files = glob.glob(os.path.join(folder, "*.gz"))
        if not gz_files:
            gz_files = glob.glob(os.path.join(folder, "part_*"))

        if not gz_files:
            continue

        # Inner Progress Bar: Files in Folder
        # leave=False removes the inner bar once the folder is finished to keep the console clean
        for gz_file in tqdm(gz_files, desc=f"   📄 Files", unit="file", leave=False):
            original_base = os.path.basename(gz_file)
            file_name = original_base.split('.')[0] + ".parquet"
            output_path = os.path.join(output_folder, file_name)

            query = f"""
            COPY (
                SELECT * FROM read_json_auto(
                    '{gz_file}', 
                    format='newline_delimited', 
                    sample_size=100000,
                    ignore_errors=True
                )
            ) TO '{output_path}' (FORMAT 'PARQUET', COMPRESSION 'SNAPPY');
            """

            try:
                con.execute(query)
                
                try:
                    os.remove(gz_file)
                except PermissionError:
                    pass 
                except Exception as e:
                    # Using tqdm.write avoids breaking the progress bar visual
                    tqdm.write(f"⚠️ Could not delete {original_base}: {e}")

            except Exception as e:
                tqdm.write(f"❌ Failed to convert: {gz_file}")
                with open(error_log_path, "a") as f:
                    f.write(f"{gz_file} | Error: {str(e)}\n")

    print("\n✅ All conversions complete!")

if __name__ == "__main__":
    run_conversion()

📁 Processing: updated_date=2025-11-06:  45%|████▌     | 158/349 [00:20<00:00, 1576.36folder/s]

In [ ]:
part_0875
